In [ ]:
from pathlib import Path

ROOT_DIR = Path("/content")
HERBFISH_DIR = ROOT_DIR/ "herbfishCV"
RAW_DATA_DIR = ROOT_DIR / "raw_data"
SAM2_DATA_DIR = ROOT_DIR / "data"

### Install SAM2 and its weights

In [ ]:
!git clone --quiet https://github.com/facebookresearch/sam2.git && cd sam2 && pip install --quiet -e ".[dev]"
!cd sam2/checkpoints && ./download_ckpts.sh

### Download herbfishCV code

In [ ]:
!git clone --quiet https://github.com/mad4octos/herbfishCV
!pip install --quiet supervision==0.26.1

### Edit YAML configuration
In `scripts/finetuning/sam2.1_hiera_large_finetune.yaml`, update the two dataset path variables to match your `SAM2_DATA_DIR`:

```yaml
img_folder: /content/data/JPEGImages   # SAM2_DATA_DIR/JPEGImages
gt_folder:  /content/data/Annotations  # SAM2_DATA_DIR/Annotations
```

### Move YAML configuration file to SAM2 dir

In [ ]:
!mv {HERBFISH_DIR}/scripts/finetuning/sam2.1_hiera_large_finetune.yaml {ROOT_DIR}/sam2/sam2/configs/sam2.1_hiera_large_finetune.yaml

### Prepare expected dir tree structure

Expected raw data dir structure:
```
{RAW_DATA_DIR}
└── <..._GX059647>/          # one folder per video, must end in GXnnnnn
    ├── images/
    │   └── train/           # JPEG frames
    │       ├── 00000.jpg
    │       ├── 00001.jpg
    │       └── ...
    └── */                   # any subdir containing COCO annotation files
        └── instances_train_vN.json   # versioned; the latest one is used
```

Output SAM2 dir structure:
```
{SAM2_DATA_DIR}
├── Annotations
│ │ 
│ ├── <video_name_1>
│ │ ├── 00000.png
│ │ ├── 00001.png
│ │ └── ...
│ │ 
│ ├── <video_name_2>
│ │ ├── 00000.png
│ │ ├── 00001.png
│ │ └── ...
│ │ 
│ ├── <video_name_...>
│ 
└── JPEGImages
  │ 
  ├── <video_name_1>
  │ ├── 00000.jpg
  │ ├── 00001.jpg
  │ └── ...
  │ 
  ├── <video_name_2>
  │ ├── 00000.jpg
  │ ├── 00001.jpg
  │ └── ...
  │ 
  └── <video_name_...>
```

In [ ]:
import re
import shutil
import subprocess

(SAM2_DATA_DIR / "JPEGImages").mkdir(exist_ok=True, parents=True)
(SAM2_DATA_DIR / "Annotations").mkdir(exist_ok=True, parents=True)

for folder in RAW_DATA_DIR.iterdir():
    if not folder.is_dir():
        continue

    print("Working on folder", folder)

    m = re.search(r"(GX\d+)$", str(folder))
    if not m:
        raise ValueError(f"Cannot extract video name from: {folder}")
    video_name = m.groups()[0]
    
    print(f"\n=== {video_name} ===")

    ###################################################################################
    # Move images to their final destination
    ###################################################################################
    src = RAW_DATA_DIR / folder / "images" / "train"
    dst = SAM2_DATA_DIR / "JPEGImages" / video_name
    if not dst.exists():
        print(f"Moving from {src} to {dst}")
        shutil.move(str(src), str(dst))

    ###################################################################################
    # Get the latest instances_train_vN.json file
    ###################################################################################
    coco_files = list((RAW_DATA_DIR / folder).glob("*/instances_train_v*.json"))
    coco_files.sort()
    coco_file = coco_files[-1]
    assert coco_file.name.startswith("instances_train_v")

    ###################################################################################
    # COCO to DAVIS
    ###################################################################################
    print("Converting COCO files to DAVIS")
    try:
        subprocess.run([
            "python", str(HERBFISH_DIR / "scripts/coco_to_sam2_masks.py"),
            "--coco-file", str(coco_file),
            "--output-dir", str(SAM2_DATA_DIR),
            "--video-name", video_name,
        ], check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(e.stdout)
        print(e.stderr)
        raise
    
    ###################################################################################
    # DAVIS padding
    ###################################################################################
    print("Padding DAVIS predictions")
    try:
        subprocess.run([
            "python", str(HERBFISH_DIR / "scripts/pad_davis_predictions.py"),
            "--gt-dir", str(SAM2_DATA_DIR / "Annotations" / video_name),
            "--images-dir", str(SAM2_DATA_DIR / "JPEGImages" / video_name),
        ], check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(e.stdout)
        print(e.stderr)
        raise

print("Completed")


### Start training

In [ ]:
!cd sam2 && python training/train.py \
    -c configs/sam2.1_hiera_large_finetune.yaml \
    --use-cluster 0 \
    --num-gpus 1